# Validation of flatcor's three default priors

This notebook presents the five studies of the validation of [flatcor](https://github.com/dm-k1/flatcor) 2.0.0,
which compares three default priors for Bayesian linear regression: flat partial *r*, flat *R* and
flat *R*², each a Beta(*a*, 1) prior on *R*² with a uniform direction. The studies follow the
order of [`docs/appendix/C_study_design.md`](docs/appendix/C_study_design.md): correctness (1–3), formal properties (4) and behaviour (5).
Every study treats the three priors the same way: the same technique, the same checks.

Each study is a script in `R/studies/` that saves its tables and checks to
`results/studies/study_NN.rds` and its figures to `figures/`; this notebook only
presents them. Every estimate carries its Monte Carlo standard error (MCSE), and every check is
judged relative to it.

**The setting below** chooses what the notebook does:

- `"results"` reads the saved results only (no computing; the default);
- `"cached"` reruns the studies first, reusing what is cached in `cache/`, so only what has
  changed is recomputed;
- `"fresh"` deletes each study's cache and reruns it from scratch.

The full validation takes hours (studies 3 and 5 most of it); [`cluster/README.md`](cluster/README.md)
describes a run on a cluster. With `smoke <- TRUE` the notebook runs or reads the smoke test
instead: a corner of each design and 10 datasets, to test the code. Its results and figures go to
`cache/`, and its checks mean nothing.

In [ ]:
mode    <- "results"   # "results", "cached" or "fresh"
smoke   <- FALSE       # TRUE: the smoke test instead of the full studies
studies <- sprintf("%02d", 1:5)    # the studies to run and show
cores   <- NA          # cores per study when running; NA: all but one

In [ ]:
## Jupyter starts in the notebook's folder, the project folder, where the scripts run too
source("R/notebook/present.R")   # run_studies(), show_study(), show_table()
stopifnot(mode %in% c("results", "cached", "fresh"))
if (mode != "results") {
  runs <- run_studies(studies, fresh = mode == "fresh", smoke = smoke, cores = cores)
  show_table(runs, "This run (exit status 1: a check failed or the study stopped; see its log)")
}
show_table(overview_of(studies, smoke), if (smoke) "Overview (smoke test)" else "Overview")

---

# Correctness

## 1. Does the code draw the prior?

flatcor's Stan model samples *R*, with *R*² ~ Beta(*a*, 1), and a uniform direction, and computes
the coefficients from them. The same four checks for each prior: the distribution of *r*² with one
predictor, of *R*² with three and with eight, and of each partial correlation, each against its
exact form by a Kolmogorov–Smirnov test at the critical value for its effective sample size.

In [ ]:
show_study("01", smoke = smoke, tables = list(
  "Sampling per setting" = function(t) data.frame(setting = names(t$iterations), iterations = t$iterations, smallest_ess = t$smallest_ess)
))

## 2. Are the Bayes factors computed correctly?

flatcor computes every Bayes factor by one route, the same for each prior: the likelihood ratio of
a model's columns averaged over *R*² ~ Beta(*a*, 1) by `integrate()`. It is checked against four
routes that share no code with it, the same for each prior: adaptive integration of a second
derivation (the prior as a mixture of g-priors), the shared likelihood ratio against its exact series
in 200-bit arithmetic, brute force, and bridge sampling on the Stan model.

In [ ]:
show_study("02", smoke = smoke, tables = list(
  "Brute force" = "brute_force",
  "Bridge sampling on the Stan fit" = "bridge_sampling"
))

## 3. Are the posterior and the Bayes factor calibrated?

Part A: simulation-based calibration (Talts et al., 2018) of the Stan model under each prior: if the
sampler is right, the rank of each true value among its posterior draws is uniform. Part B: on data
drawn from each prior, half from the model without *x*₁, the share without *x*₁ among datasets with a
given BF₁₀ equals 1 / (1 + BF₁₀) (Schad et al., 2023); on data from JZS it should not.

In [ ]:
show_study("03", smoke = smoke, tables = list(
  "Rank uniformity (chi-square over 20 bins)" = "sbc_uniformity",
  "Calibration of the Bayes factor per generating prior and test" = "calibration_tests"
))

---

# Formal properties

## 4. Does each prior meet the objective-Bayes criteria?

The criteria of Bayarri, Berger, Forte and García-Donato (2012) for each prior: predictive
matching, information consistency, intrinsic consistency, model-selection consistency for a fixed
and a growing number of predictors *k*, and invariance under rescaling and recoding.

In [ ]:
show_study("04", smoke = smoke, tables = list(
  "Rate of the log Bayes factor in log n" = "rate_slopes",
  "Median log Bayes factor under the alternative" = "alternative_medians",
  "Growing k: rate in log n" = "growing_slopes"
))

---

# Behaviour

## 5. How does each prior behave on data with a known truth?

The three priors next to JZS and Zellner–Siow, on the same datasets. Part A: how often the Bayes
factor for *x*₁ (the model with *x*₁ against the model without it) supports either model or stays
undecided, how that depends on the number of predictors *k*, and how well it separates a null from
a small effect. Part B: the same for a factor. Part C: bias, interval coverage and prediction error
of the posterior.

In [ ]:
show_study("05", smoke = smoke, tables = list(
  "One coefficient: share of each answer, pooled over the cells at each k" = function(t) answers_side_by_side(t$shares_per_k, c("k", "slope_x1", "method")),
  "One coefficient: true nulls confirmed at k = 2 and k = 20" = "k_drift",
  "One coefficient: separating slope 0 from slope 0.1, AUC per design" = "auc",
  "Factor: share of each answer, averaged over q" = function(t) answers_side_by_side(t$factor_shares_over_q, c("n", "other_predictors", "factor_effect", "method")),
  "Factor: separating a null factor from a small one, AUC" = "factor_auc",
  "Estimation: bias and coverage of the slope of x1" = "bias_and_coverage",
  "Estimation: prediction error relative to OLS" = "prediction_ratio"
))